# Summary of evaluation of models on CORDEX-ML_BENCH datasets

In [ ]:
%reload_ext autoreload

%autoreload 2

%reload_ext dotenv
%dotenv

In [ ]:
from mlde_analysis.cordex_ml_default_params import *

In [ ]:
import functools
import math
import string

import IPython
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from properscoring import crps_ensemble
import xarray as xr

from mlde_analysis import plot_map, SUBREGIONS, BOX_LOCATIONS
from mlde_analysis.display import pretty_table, VAR_RANGES
from mlde_analysis.distribution import mean_bias, std_bias, stat_bias, plot_freq_density, plot_distribution_figure, compute_metrics, DIST_THRESHOLDS, plot_freq_density_figure
# from mlde_analysis.wet_dry import threshold_exceeded_prop_stats, threshold_exceeded_prop, threshold_exceeded_prop_error, threshold_exceeded_prop_change, plot_threshold_exceedence_errors, THRESHOLDS, wd_mean, wd_mean_bias
from mlde_analysis.psd import plot_psd, pysteps_rapsd
from mlde_utils import cp_model_rotated_pole
from mlde_analysis import qq_plot, reasonable_quantiles

In [ ]:
matplotlib.rcParams['figure.dpi'] = 300

In [ ]:
IPython.display.Markdown(desc)

In [ ]:
%reload_ext mlde_analysis.magics 
EVAL_DS, MODELS, TARGET_DAS, PRED_DAS, VAR_DAS, MODELLABEL2SPEC = %load_eval_data
EVAL_DS

## Figure: distribution

* Frequency Density Histogram of rainfall intensities
* Maps of Mean bias ($\frac{\mu_{sample}-\mu_{CPM}}{\mu_{CPM}}$) over all samples, time and ensemble members
* Std Dev Bias $\frac{\sigma_{sample}}{\sigma_{CPM}}$ over all samples, time and ensemble members

Table of:

* RMS biases
* J-S Distances
* proportion of density over thresholds

In [ ]:
for var in eval_vars:
    IPython.display.display_markdown(f"### {var}", raw=True)
    
    hist_das = PRED_DAS[var]
    target_da = TARGET_DAS[var]
    
    metrics_ds = compute_metrics(hist_das, target_da, thresholds=DIST_THRESHOLDS[var])

    pretty_table(metrics_ds, round=4)
    
    normalize=(var == "pr")
    mean_biases = PRED_DAS[var].groupby("model").map(mean_bias, target_da=target_da, normalize=(var=="pr"))
    
    bias_kwargs = {"style": f"{var}Bias"}
    for fd_kwargs in [{"yscale": "log", "target_label": target_sim_key}, {"yscale": "linear", "target_label": target_sim_key}]:
        if fd_kwargs["yscale"] == "linear" and var == "pr":
            continue
        fig = plt.figure(layout="constrained", figsize=(5.5, 4))
        
        axd = plot_distribution_figure(
            fig,
            hist_das,
            target_da,
            {"meanb": mean_biases},
            MODELLABEL2SPEC,
            hrange=VAR_RANGES[var], fd_kwargs=fd_kwargs, bias_kwargs=bias_kwargs
        )
        if var == "relhum150cm":
            axd["Density"].axvline(x=100, color='k', linestyle='--', linewidth=1)
        
        plt.show()

## Figures: Seasonal distribution

* Frequency Density Histogram of rainfall intensities
* Maps of Mean bias ($\frac{\mu_{sample}-\mu_{CPM}}{\mu_{CPM}}$) over all samples, time and ensemble members
* Std Dev Bias $\frac{\sigma_{sample}}{\sigma_{CPM}}$ over all samples, time and ensemble members

In [ ]:
for var in eval_vars:
    IPython.display.display_markdown(f"### {var}", raw=True)

    metrics_ds = VAR_DAS[var].groupby("time.season").map(lambda season_ds:compute_metrics(season_ds[f"pred_{var}"], season_ds[f"target_{var}"], thresholds=DIST_THRESHOLDS[var]))

    pretty_table(metrics_ds, round=4, dim_order=["season", "model"])
    
    for season, season_ds in VAR_DAS[var].groupby("time.season"):
        IPython.display.display_markdown(f"#### {season}", raw=True)
        hist_das = season_ds[f"pred_{var}"]
        target_da = season_ds[f"target_{var}"]
        normalize=(var == "pr")
        mean_biases = season_ds[f"pred_{var}"].groupby("model").map(mean_bias, target_da=target_da, normalize=normalize)
        
        bias_kwargs = {"style": f"{var}Bias"}
        for fd_kwargs in [{"yscale": "log"}, {"yscale": "linear"}]:
            if var == "pr" and fd_kwargs["yscale"] == "linear":
                continue
            fig = plt.figure(layout="constrained", figsize=(5.5, 6.5))
            axd = plot_distribution_figure(
                fig,
                hist_das,
                target_da,
                {"meanb": mean_biases,},
                MODELLABEL2SPEC, 
                hrange=VAR_RANGES[var],
                fd_kwargs=fd_kwargs,
                bias_kwargs=bias_kwargs,
            )
            if var == "relhum150cm":
                axd["Density"].axvline(x=100, color='k', linestyle='--', linewidth=1)
            
        plt.show()

## Figure: structure

* PSD

In [ ]:
# if len(eval_vars) > 1:
#     gridspec = np.pad(np.array(eval_vars), (0, -len(eval_vars) % 2), constant_values=".").reshape(-1, 1)
# else:
#     gridspec = np.array([eval_vars])
# structure_fig = plt.figure(figsize=(5.5*gridspec.shape[1], 3.5*gridspec.shape[0]), layout="constrained")
# axd = structure_fig.subplot_mosaic(gridspec, sharey=True, sharex=False)

for var in eval_vars:
    IPython.display.display_markdown(f"### {var}", raw=True)
    # gridspec = np.array([var]).reshape(1,1)
    structure_fig = plt.figure(figsize=(4, 3), layout="constrained")
    axd = structure_fig.subplot_mosaic([[var]], sharey=True, sharex=False)
    target_hr_rapsd = pysteps_rapsd(
        TARGET_DAS[var].
            stack(example=["ensemble_member", "time"]).cf.
            transpose("example", "Y", "X"),
        pixel_size=8.8
    ).mean(dim="example").drop_sel(freq=0)
    
    pred_rapsds = [
        {
            "label": model,
            "color": spec["color"],
            "data": pysteps_rapsd(
                EVAL_DS[source][f"pred_{var}"].sel(model=model).
                    stack(example=["ensemble_member", "sample_id", "time"]).cf.
                    transpose("example", "Y", "X"), 
                pixel_size=8.8).mean(dim="example").drop_sel(freq=0)
        }
        for source, mconfigs in MODELS.items() for model, spec in mconfigs.items()
    ]
    
    ax = axd[var]

    plot_psd(target_hr_rapsd, pred_rapsds, ax=ax)
    # ax.set_title(CPM_DAS[var].attrs["long_name"])
    
    plt.show()

    rapsd_errors_da = xr.concat([ np.abs(100*(h["data"] - target_hr_rapsd)/target_hr_rapsd) for h in pred_rapsds ], dim="model").rename("raspd_error").assign_coords(wavelength=(["freq"], 1/target_hr_rapsd["freq"].values))

    fig = plt.figure(figsize=(4, 3), layout="constrained")
    ax = fig.subplots(1)
    l = rapsd_errors_da.plot(x="wavelength", hue="model", ax=ax)
    ax.set_title("RAPSD % error", fontsize="small")
    ax.set_xlim(10, 1000)
    ax.set_xscale("log")
    plt.rc('legend', fontsize = "xx-small", title_fontsize="x-small")
    
    plt.show()
    
    _ = pretty_table(rapsd_errors_da, round=4, dim_order=["freq", "model"])

## QQ plots

In [ ]:
quantile_dims=["ensemble_member", "T", "X", "Y"]

for var in eval_vars:
    IPython.display.display_markdown(f"### {var}", raw=True)

    target_da = TARGET_DAS[var]
    quantiles = reasonable_quantiles(target_da)
    target_quantiles = target_da.cf.quantile(quantiles, dim=quantile_dims).rename("target_q")

    for source, ds in EVAL_DS.items():
        pred_da = ds[f"pred_{var}"]
        pred_quantiles = pred_da.cf.quantile(quantiles, dim=quantile_dims).rename("pred_q")

        layout="constrained"

        fig, ax = plt.subplots(figsize=(3.5, 3.5), layout="constrained")

        xlabel = f"CPM \n{xr.plot.utils.label_from_attrs(da=target_da)}"
        ylabel = f"Predicted \n{xr.plot.utils.label_from_attrs(da=pred_da)}"

        qq_plot(ax, target_quantiles, pred_quantiles, title=f"Predicted quantiles vs Target quantiles", xlabel=xlabel, ylabel=ylabel)

    plt.show()

### Table: correlations

* pred domain mean & ensemble mean vs target domain mean
* pred domain mean vs target domain mean
* pred ensemble mean vs target
* pred vs target

In [ ]:
mois = { model: mconfig for mconfigs in MODELS.values() for model, mconfig in mconfigs.items() if mconfig.get("UQ", True) }

corr_coeff_das = []

for var in eval_vars:
    ds = VAR_DAS[var].sel(model=list(mois.keys()))

    pred_da = ds[f"pred_{var}"]
    target_da = ds[f"target_{var}"]
    
    corr_coeff_das.append(
        xr.merge([
            xr.corr(pred_da.cf.mean(dim=["Y", "X", "sample_id"]), target_da.cf.mean(dim=["Y", "X"]), dim=["ensemble_member", "time"]).rename(f"domain and sample mean corr"),
            xr.corr(pred_da.cf.mean(dim=["Y", "X"]), target_da.cf.mean(dim=["Y", "X"]), dim=["sample_id", "ensemble_member", "time"]).rename(f"domain mean corr (pt per sample)"),
            xr.corr(pred_da.mean(dim=["sample_id"]), target_da, dim=["ensemble_member", "time", target_da.cf["Y"].name, target_da.cf["X"].name,]).rename(f"sample mean corr (pt per gridbox)"),
            xr.corr(pred_da, target_da, dim=["ensemble_member", "time", target_da.cf["Y"].name, target_da.cf["X"].name, "sample_id"]).rename(f"corr (pt per gridbox and sample)"),
            xr.corr(pred_da.mean(dim=["sample_id"]).cf.max(dim=["Y", "X"]), target_da.cf.max(dim=["Y", "X"]), dim=["ensemble_member", "time"]).rename(f"sample mean domain max corr"),
            xr.corr(pred_da.cf.max(dim=["Y", "X", "sample_id"]), target_da.cf.max(dim=["Y", "X"]), dim=["ensemble_member", "time"]).rename(f"domain and sample max corr"),
            xr.corr(pred_da.cf.max(dim=["Y", "X"]), target_da.cf.max(dim=["Y", "X"]), dim=["ensemble_member", "time", "sample_id"]).rename(f"domain max corr (pt per sample)"),
        ]).expand_dims({"var": [var]})
    )

_ = pretty_table(xr.concat(corr_coeff_das, dim="var"), round=2)

## CRPS

In [ ]:
def group_crps(model_forecast_da, truth_da):
    return xr.apply_ufunc(
        crps_ensemble,
        truth_da,
        model_forecast_da.squeeze("model"),
        input_core_dims=[truth_da.dims, model_forecast_da.squeeze("model").dims],  # list with one entry per arg
        output_core_dims=[["examples", "grid_latitude", "grid_longitude"]],
        # vectorize=True,
    ).rename("CRPS").mean()

for var in eval_vars:
    print(var)
    
    mois = { model: mconfig for mconfigs in MODELS.values() for model, mconfig in mconfigs.items() if mconfig.get("UQ", True) }

    forecasts_da = PRED_DAS[var].sel(model=list(mois.keys())).stack(example=["ensemble_member", "time"]).cf.transpose("model", "example", "Y", "X", "sample_id") 
    crps_scores = {}

    truth = TARGET_DAS[var].stack(example=["ensemble_member", "time"]).cf.transpose("example", "Y", "X")
    
    crps_scores = forecasts_da.groupby("model", squeeze=False).map(group_crps, truth_da=truth)
    pretty_table(crps_scores, round=4)